# Dataset structure statistics

This notebook scans the dataset folder and reports case-level, side-level, visit-level, and image-level statistics.

In [4]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path(r"G:\wxy_codes\dataset_hydro_123\raw")
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".dcm"}

assert ROOT.exists(), f"Dataset root does not exist: {ROOT}"

In [5]:
case_rows = []
side_rows = []
visit_rows = []
image_rows = []

case_dirs = sorted([p for p in ROOT.iterdir() if p.is_dir() and p.name.startswith("case_")])

for case_dir in case_dirs:
    case_meta_path = case_dir / "case_meta.json"
    case_meta = json.loads(case_meta_path.read_text(encoding="utf-8")) if case_meta_path.exists() else {}
    case_id = case_meta.get("case_id", case_dir.name)
    left_label = int(case_meta.get("left_label", 0))
    right_label = int(case_meta.get("right_label", 0))
    case_label = 1 if (left_label == 1 or right_label == 1) else 0
    case_visit_count = 0
    case_image_count = 0

    for side, label in [("left", left_label), ("right", right_label)]:
        side_rows.append({
            "case_id": case_id,
            "side": side,
            "label": label,
            "label_name": "positive" if label == 1 else "negative",
        })

    for phase_dir in sorted([p for p in case_dir.iterdir() if p.is_dir()]):
        if phase_dir.name not in {"1prenatal", "2postnatal"}:
            continue

        for visit_dir in sorted([p for p in phase_dir.iterdir() if p.is_dir()]):
            visit_meta_path = visit_dir / "meta.json"
            visit_meta = json.loads(visit_meta_path.read_text(encoding="utf-8")) if visit_meta_path.exists() else {}
            visit_id = visit_meta.get("visit_id", visit_dir.name)
            phase = visit_meta.get("phase", phase_dir.name)
            exam_date = visit_meta.get("exam_date")
            left_image_count = 0
            right_image_count = 0

            for side in ["left", "right"]:
                image_dir = visit_dir / side / "images"
                images = sorted([p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]) if image_dir.exists() else []
                if side == "left":
                    left_image_count = len(images)
                else:
                    right_image_count = len(images)

                for image_path in images:
                    image_rows.append({
                        "case_id": case_id,
                        "visit_id": visit_id,
                        "phase": phase,
                        "exam_date": exam_date,
                        "side": side,
                        "image_name": image_path.name,
                        "image_path": str(image_path),
                    })

            total_image_count = left_image_count + right_image_count
            case_visit_count += 1
            case_image_count += total_image_count
            visit_rows.append({
                "case_id": case_id,
                "visit_id": visit_id,
                "phase": phase,
                "exam_date": exam_date,
                "left_image_count": left_image_count,
                "right_image_count": right_image_count,
                "total_image_count": total_image_count,
            })

    case_rows.append({
        "case_id": case_id,
        "left_label": left_label,
        "right_label": right_label,
        "case_label": case_label,
        "case_label_name": "positive" if case_label == 1 else "negative",
        "visit_count": case_visit_count,
        "image_count": case_image_count,
    })

case_df = pd.DataFrame(case_rows)
side_df = pd.DataFrame(side_rows)
visit_df = pd.DataFrame(visit_rows)
image_df = pd.DataFrame(image_rows)

visit_df["exam_date"] = pd.to_datetime(visit_df["exam_date"], errors="coerce")
image_df["exam_date"] = pd.to_datetime(image_df["exam_date"], errors="coerce")

case_df.head(), visit_df.head(), image_df.head()

(     case_id  left_label  right_label  case_label case_label_name  \
 0  case_0001           1            0           1        positive   
 1  case_0002           1            0           1        positive   
 2  case_0003           0            1           1        positive   
 3  case_0004           0            1           1        positive   
 4  case_0005           0            1           1        positive   
 
    visit_count  image_count  
 0            7           57  
 1           10           82  
 2           13           83  
 3           13           71  
 4            6           61  ,
      case_id      visit_id     phase  exam_date  left_image_count  \
 0  case_0001  202412300188  prenatal 2024-12-30                 4   
 1  case_0001  202503081263  prenatal 2025-03-08                 2   
 2  case_0001  202503310610  prenatal 2025-03-31                 9   
 3  case_0001  202504040581  prenatal 2025-04-04                 2   
 4  case_0001  202504111592  prenatal 202

In [6]:
# Overall answers
case_label_counts = case_df["case_label_name"].value_counts().reindex(["positive", "negative"], fill_value=0)
side_label_counts = side_df["label_name"].value_counts().reindex(["positive", "negative"], fill_value=0)

years = sorted(visit_df["exam_date"].dropna().dt.year.unique().tolist())
year_span = f"{min(years)}-{max(years)}" if years else "unknown"

summary = pd.DataFrame([
    {"item": "total_cases", "value": len(case_df), "note": "case_xxxx directories"},
    {"item": "case_positive", "value": int(case_label_counts["positive"]), "note": "case-level: left_label=1 or right_label=1"},
    {"item": "case_negative", "value": int(case_label_counts["negative"]), "note": "case-level: both left_label=0 and right_label=0"},
    {"item": "side_positive", "value": int(side_label_counts["positive"]), "note": "side-level: left/right counted separately"},
    {"item": "side_negative", "value": int(side_label_counts["negative"]), "note": "side-level: left/right counted separately"},
    {"item": "total_visits", "value": len(visit_df), "note": "visit/exam directories"},
    {"item": "total_images", "value": len(image_df), "note": "image files under left/right/images"},
    {"item": "year_span", "value": year_span, "note": f"years found: {years}"},
    {"item": "center_count", "value": 1, "note": "single-center, based on your provided description"},
])

summary

,item,value,note
0,total_cases,106,case_xxxx directories
1,case_positive,71,case-level: left_label=1 or right_label=1
2,case_negative,35,case-level: both left_label=0 and right_label=0
3,side_positive,77,side-level: left/right counted separately
4,side_negative,135,side-level: left/right counted separately
5,total_visits,761,visit/exam directories
6,total_images,3555,image files under left/right/images
7,year_span,2019-2026,"years found: [2019, 2020, 2021, 2022, 2023, 20..."
8,center_count,1,"single-center, based on your provided description"


In [7]:
# How many images and visits does each patient/case have?
case_df.sort_values("case_id")

,case_id,left_label,right_label,case_label,case_label_name,visit_count,image_count
0,case_0001,1,0,1,positive,7,57
1,case_0002,1,0,1,positive,10,82
2,case_0003,0,1,1,positive,13,83
3,case_0004,0,1,1,positive,13,71
4,case_0005,0,1,1,positive,6,61
...,...,...,...,...,...,...,...
101,case_0164,0,0,0,negative,9,25
102,case_0165,0,0,0,negative,10,25
103,case_0166,0,0,0,negative,5,10
104,case_0167,0,0,0,negative,8,19


In [8]:
# How many images does each visit/exam have?
visit_df.sort_values(["case_id", "exam_date", "visit_id"])

,case_id,visit_id,phase,exam_date,left_image_count,right_image_count,total_image_count
0,case_0001,202412300188,prenatal,2024-12-30,4,5,9
1,case_0001,202503081263,prenatal,2025-03-08,2,2,4
2,case_0001,202503310610,prenatal,2025-03-31,9,4,13
3,case_0001,202504040581,prenatal,2025-04-04,2,4,6
4,case_0001,202504111592,prenatal,2025-04-11,3,3,6
...,...,...,...,...,...,...,...
756,case_0168,202502081638,prenatal,2025-02-08,2,0,2
757,case_0168,202503101028,prenatal,2025-03-10,1,1,2
758,case_0168,202503200244,prenatal,2025-03-20,0,0,0
759,case_0168,202503211887,postnatal,2025-03-21,2,2,4


In [9]:
# Helpful distributions for reporting ranges/median/IQR if needed
distributions = {
    "images_per_case": case_df["image_count"].describe(),
    "visits_per_case": case_df["visit_count"].describe(),
    "images_per_visit": visit_df["total_image_count"].describe(),
}

for name, stats in distributions.items():
    print(f"\n{name}")
    print(stats)

zero_image_visits = visit_df[visit_df["total_image_count"] == 0]
print(f"\nzero_image_visits: {len(zero_image_visits)}")
display(zero_image_visits.sort_values(["case_id", "exam_date", "visit_id"]))


images_per_case
count    106.000000
mean      33.537736
std       27.639232
min        1.000000
25%       12.000000
50%       23.000000
75%       51.750000
max      111.000000
Name: image_count, dtype: float64

visits_per_case
count    106.000000
mean       7.179245
std        4.194775
min        1.000000
25%        4.000000
50%        6.000000
75%       10.000000
max       24.000000
Name: visit_count, dtype: float64

images_per_visit
count    761.000000
mean       4.671485
std        3.095385
min        0.000000
25%        2.000000
50%        4.000000
75%        6.000000
max       29.000000
Name: total_image_count, dtype: float64

zero_image_visits: 51


,case_id,visit_id,phase,exam_date,left_image_count,right_image_count,total_image_count
91,case_0010,202501310683,prenatal,2025-01-31,0,0,0
102,case_0011,20260115001173,postnatal,2026-01-15,0,0,0
120,case_0015,202502261746,postnatal,2025-02-26,0,0,0
128,case_0016,20241118000286,prenatal,2024-11-18,0,0,0
140,case_0018,20240729003382,prenatal,2024-07-29,0,0,0
169,case_0021,20240314003160,postnatal,2024-03-14,0,0,0
205,case_0024,20231217001142,postnatal,2023-12-17,0,0,0
210,case_0025,20221203000677,postnatal,2022-12-03,0,0,0
255,case_0030,20220531001896,postnatal,2022-05-31,0,0,0
274,case_0032,20220422001817,postnatal,2022-04-22,0,0,0


In [10]:
# Optional: save detail tables next to the notebook
OUT_DIR = Path.cwd() / "dataset_statistics_outputs"
OUT_DIR.mkdir(exist_ok=True)

summary.to_csv(OUT_DIR / "summary.csv", index=False, encoding="utf-8-sig")
case_df.to_csv(OUT_DIR / "case_level_statistics.csv", index=False, encoding="utf-8-sig")
visit_df.to_csv(OUT_DIR / "visit_level_statistics.csv", index=False, encoding="utf-8-sig")
side_df.to_csv(OUT_DIR / "side_level_labels.csv", index=False, encoding="utf-8-sig")

OUT_DIR

WindowsPath('g:/wxy_codes/Fetal_hydronephrosis/dataset_statistics/dataset_statistics_outputs')